In [1]:
import pandas as pd
import numpy as np
import optuna
from plotly.io import show
import optuna.visualization as ov
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.inspection import DecisionBoundaryDisplay
import re

In [2]:
#importing datasets
df = pd.read_csv("train_titanic.csv")

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [3]:
df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [4]:
df = df.drop(columns=['Cabin', 'Name'])

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Sex          891 non-null    object 
 4   Age          714 non-null    float64
 5   SibSp        891 non-null    int64  
 6   Parch        891 non-null    int64  
 7   Ticket       891 non-null    object 
 8   Fare         891 non-null    float64
 9   Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(3)
memory usage: 69.7+ KB


In [5]:
def extract_numeric(x):
    if pd.isna(x):
        return np.nan
    s = str(x)
    m = re.findall(r"[-+]?\d*\.?\d+", s)
    if not m:
        return np.nan
    return float(m[0])

numeric_cols_raw = [
    "Ticket",
]

for col in numeric_cols_raw:
    if col in df.columns:
        df[col] = df[col].apply(extract_numeric)
        
df["Age"] = df["Age"].fillna(df["Age"].median())

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Sex          891 non-null    object 
 4   Age          891 non-null    float64
 5   SibSp        891 non-null    int64  
 6   Parch        891 non-null    int64  
 7   Ticket       887 non-null    float64
 8   Fare         891 non-null    float64
 9   Embarked     889 non-null    object 
dtypes: float64(3), int64(5), object(2)
memory usage: 69.7+ KB


In [6]:
#remove outlier
numeric_cols = df.select_dtypes(include=["float", "int"]).columns.tolist()

for col in numeric_cols:
    Q1 = df[col].quantile(0.2)
    Q3 = df[col].quantile(0.8)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

    df[col] = np.where(df[col].between(lower, upper), df[col], np.nan)

df = df.dropna()

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 741 entries, 0 to 890
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  741 non-null    float64
 1   Survived     741 non-null    float64
 2   Pclass       741 non-null    float64
 3   Sex          741 non-null    object 
 4   Age          741 non-null    float64
 5   SibSp        741 non-null    float64
 6   Parch        741 non-null    float64
 7   Ticket       741 non-null    float64
 8   Fare         741 non-null    float64
 9   Embarked     741 non-null    object 
dtypes: float64(8), object(2)
memory usage: 63.7+ KB


In [7]:
df = pd.get_dummies(df, drop_first=True)

In [8]:
def objective(trial):
    params = {
        'objective': 'binary:logistic',
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000), # Intero tra 100 e 1000
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.3, log=True), # Scala logaritmica
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0), # % feature per albero
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True), # L1 Reg (Lasso)
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True), # L2 Reg (Ridge)
        'n_jobs': 3,
        'random_state': 42
    }
    
    model = xgb.XGBClassifier(**params)
    
    # We use 5-fold cross-validation to ensure that the result isn't just a fluke
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
    
    if trial.should_prune():
            raise optuna.TrialPruned()
    
    # Return the average accuracy
    return scores.mean()

In [9]:
X = df.drop('Survived', axis=1)
y = df["Survived"].astype(int)

In [10]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

**OPTUNA**

In [12]:
sampler = optuna.samplers.TPESampler(seed=10)
study = optuna.create_study(
    direction="maximize",
    sampler=sampler,
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=5, n_warmup_steps=30, interval_steps=10
    ),
)

study.optimize(objective, n_trials=50, show_progress_bar=True)

print("\n--- Optimization Results ---")
print(f"Best Trial (Attempt #{study.best_trial.number})")
print(f"Best Accuracy (CV): {study.best_value:.4f}")
print("Best Hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

best_params = study.best_params

final_model = xgb.XGBClassifier(**best_params, n_jobs=3, random_state=42)
final_model.fit(X_train, y_train)

final_acc = final_model.score(X_test, y_test)

print(f"\nFinal Accuracy on Test Set: {final_acc:.4f}")

[I 2026-03-27 22:16:48,564] A new study created in memory with name: no-name-6e71f0b7-93da-47ce-be9a-4fcf40029622


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-03-27 22:16:49,920] Trial 0 finished with value: 0.8108246688505911 and parameters: {'n_estimators': 794, 'learning_rate': 0.00112565445587888, 'max_depth': 8, 'subsample': 0.8744019412693059, 'colsample_bytree': 0.7492535061512953, 'reg_alpha': 1.0547992438188775e-06, 'reg_lambda': 6.061300044367956e-07}. Best is trial 0 with value: 0.8108246688505911.
[I 2026-03-27 22:16:50,342] Trial 1 finished with value: 0.8056971941318901 and parameters: {'n_estimators': 785, 'learning_rate': 0.00262366298138214, 'max_depth': 3, 'subsample': 0.8426799091838986, 'colsample_bytree': 0.9766966730974682, 'reg_alpha': 1.08526150100961e-08, 'reg_lambda': 0.0004071274363191098}. Best is trial 0 with value: 0.8108246688505911.
[I 2026-03-27 22:16:51,084] Trial 2 finished with value: 0.7938755163082182 and parameters: {'n_estimators': 832, 'learning_rate': 0.032907988677835835, 'max_depth': 8, 'subsample': 0.6459380340853166, 'colsample_bytree': 0.9588870612564717, 'reg_alpha': 0.02698870526662561

Without pruning: Final Accuracy on the Test Set: 0.8456

With pruning: --- Optimization Results ---
- Best Trial (Trial #26)
- Best Accuracy (CV): 0.8159
- Best Hyperparameters:
  - n_estimators: 380
  - learning_rate: 0.005295232368540756
  - max_depth: 7
  - subsample: 0.6891247712682177
  - colsample_bytree: 0.6752990656146991
  - reg_alpha: 0.06532885224059053
  - reg_lambda: 0.00013423432607369992

- Final Accuracy on Test Set: 0.8591

Outliers 0.2, 0.8: --- Optimization Results ---
- Best Trial (Attempt #35)
- Best Accuracy (CV): 0.8192
- Best Hyperparameters:
  - n_estimators: 821
  - learning_rate: 0.0040452630812914565
  - max_depth: 6
  - subsample: 0.8144802291812205
  - colsample_bytree: 0.8504236185894885
  - reg_alpha: 0.0020024465144113522
  - reg_lambda: 0.013159768501168314

- Final Accuracy on the Test Set: 0.8523

**PLOT**

In [13]:
ov.plot_optimization_history(study).show()
ov.plot_param_importances(study).show()
ov.plot_parallel_coordinate(study).show()
ov.plot_contour(study).show()
ov.plot_intermediate_values(study).show()

[W 2026-03-27 22:17:08,939] You need to set up the pruning feature to utilize `plot_intermediate_values()`
